# SurveyCTO GeoJSON Test

Generate 200m grid cells around office locations and export as GeoJSON for SurveyCTO testing.

In [ ]:
import geopandas as gpd
import json
from shapely.geometry import Polygon, box, mapping
from pathlib import Path


def make_shapes(cx: float, cy: float, half: float) -> dict:
    """Return {shape_name: shapely polygon} for one office cell in local UTM."""
    square = box(cx - half, cy - half, cx + half, cy + half)

    # Chevron: square with the left edge pushed inward to a point at the centre.
    # Non-symmetric so enumerators can tell left from right at a glance.
    chevron = Polygon([
        (cx, cy),                    # left point
        (cx - half, cy + half),      # top-left
        (cx + half, cy + half),      # top-right
        (cx + half, cy - half),      # bottom-right
        (cx - half, cy - half),      # bottom-left
    ])

    return {"square": square, "chevron": chevron}


SHAPES = ("square", "chevron")

In [2]:
# Office locations (lat, lon)
offices = {
    "dar_es_salaam": (-6.778436239590647, 39.25134881249597),
    "kampala": (0.3421994896849769, 32.59192784637421), 
    "amsterdam": (52.37297325089347, 4.887482952303158)  # <-- add coordinates here
}

GRID_SIZE = 200  # metres
OUTPUT_DIR = Path(r"G:\Shared drives\TZ-CCT_RUBEV-0825\Data\0_Listing\1_Input\surveycto_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for name, (lat, lon) in offices.items():
    # Create point in WGS84, project to local UTM for a clean metric buffer
    point = gpd.GeoSeries.from_xy([lon], [lat], crs="EPSG:4326")
    utm_crs = point.estimate_utm_crs()
    point_utm = point.to_crs(utm_crs)

    cx, cy = point_utm.iloc[0].x, point_utm.iloc[0].y
    half = GRID_SIZE / 2

    # Emit one GeoJSON per shape per office
    for shape_name, polygon in make_shapes(cx, cy, half).items():
        gdf = gpd.GeoDataFrame({"name": [name]}, geometry=[polygon], crs=utm_crs)
        gdf_wgs84 = gdf.to_crs(epsg=4326)

        centroid = gdf_wgs84.geometry.iloc[0].centroid

        geojson = {
            "type": "FeatureCollection",
            "features": [{
                "type": "Feature",
                "properties": {
                    "name": name,
                    "shape": shape_name,
                    "grid_size_m": GRID_SIZE,
                    "centroid_lat": centroid.y,
                    "centroid_lon": centroid.x,
                },
                "geometry": mapping(gdf_wgs84.geometry.iloc[0]),
            }],
        }

        out_path = OUTPUT_DIR / f"{name}_{shape_name}_{GRID_SIZE}m.geojson"
        out_path.write_text(json.dumps(geojson, indent=2))
        print(f"{name} ({shape_name}): saved to {out_path.name}")
        print(f"  Centroid: {centroid.y:.6f}, {centroid.x:.6f}")
    print(f"  UTM CRS: {utm_crs}")
    print()

In [ ]:
# Generate a static map with the grid overlaid on satellite imagery
import contextily as cx
import matplotlib.pyplot as plt

for name, (lat, lon) in offices.items():
    for shape_name in SHAPES:
        geojson_path = OUTPUT_DIR / f"{name}_{shape_name}_{GRID_SIZE}m.geojson"
        gdf = gpd.read_file(geojson_path)
        gdf_wm = gdf.to_crs(epsg=3857)

        fig, ax = plt.subplots(figsize=(10, 10))
        gdf_wm.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=3)

        # Buffer view
        bounds = gdf_wm.total_bounds
        dx = (bounds[2] - bounds[0]) * 0.5
        dy = (bounds[3] - bounds[1]) * 0.5
        ax.set_xlim(bounds[0] - dx, bounds[2] + dx)
        ax.set_ylim(bounds[1] - dy, bounds[3] + dy)

        cx.add_basemap(ax, source="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}", zoom=18)

        ax.set_title(f"{name} ({shape_name}) — {GRID_SIZE}m grid cell", fontsize=14, fontweight="bold")
        ax.set_axis_off()

        map_path = OUTPUT_DIR / f"{name}_{shape_name}_{GRID_SIZE}m_map.png"
        fig.savefig(map_path, dpi=150, bbox_inches="tight")
        print(f"Saved map: {map_path.name}")
        plt.show()

In [ ]:
# Interactive Folium map (saved as HTML) — one per shape per office
import folium

for name, (lat, lon) in offices.items():
    for shape_name in SHAPES:
        geojson_path = OUTPUT_DIR / f"{name}_{shape_name}_{GRID_SIZE}m.geojson"
        geojson_data = json.loads(geojson_path.read_text())

        m = folium.Map(location=[lat, lon], zoom_start=17, tiles="OpenStreetMap")
        folium.TileLayer(
            tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
            attr="Google Hybrid", name="Google Hybrid"
        ).add_to(m)
        folium.GeoJson(geojson_data, style_function=lambda x: {
            "color": "red", "weight": 3, "fillOpacity": 0.2
        }).add_to(m)
        folium.Marker([lat, lon], popup=f"{name} ({shape_name})").add_to(m)
        folium.LayerControl().add_to(m)

        html_path = OUTPUT_DIR / f"{name}_{shape_name}_{GRID_SIZE}m_map.html"
        m.save(str(html_path))
        print(f"Saved interactive map: {html_path.name}")
        display(m)

## Production-style MBTiles

Generate `.mbtiles` files for the same office cells using the production helper [`src/mapping/mbtiles_export.py`](../src/mapping/mbtiles_export.py) — the same function `scripts/generate_all_maps.py` calls when run with `--mbtiles`.

This is what SurveyCTO will actually load: a self-contained offline basemap with the cell boundary, centroid crosshair, and a title banner burned in. One MBTiles file is written per shape (square + chevron) per office. No `building_count` or `selection_role` is required for these test cells; we pass `role="primary"` to get the green outline.

In [ ]:
import sys
from pathlib import Path

# Make the project src/ importable when running this notebook standalone
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.mapping.mbtiles_export import export_detail_mbtiles

for name, (lat, lon) in offices.items():
    for shape_name in SHAPES:
        geojson_path = OUTPUT_DIR / f"{name}_{shape_name}_{GRID_SIZE}m.geojson"
        cell_gdf = gpd.read_file(geojson_path)  # already WGS84

        centroid = cell_gdf.geometry.iloc[0].centroid
        title = f"{name.replace('_', ' ').title()} ({shape_name}) — {GRID_SIZE}m test cell"

        mbtiles_path = OUTPUT_DIR / f"{name}_{shape_name}_{GRID_SIZE}m.mbtiles"
        export_detail_mbtiles(
            subcell=cell_gdf,
            output_path=mbtiles_path,
            buildings=None,        # office cells: no building footprints layer
            role="primary",        # picks the green outline
            zoom=19,
            title=title,
        )

        print(f"{name} ({shape_name}):")
        print(f"  Wrote: {mbtiles_path.name}")
        print(f"  Size:  {mbtiles_path.stat().st_size / 1e6:.2f} MB")
        print(f"  Centroid (WGS84): {centroid.y:.6f}, {centroid.x:.6f}")
    print()

In [ ]:
# Visual verification — read each .mbtiles back and overlay it on a folium map
import rasterio
from rasterio.warp import transform_bounds
from folium.raster_layers import ImageOverlay

for name in offices:
    for shape_name in SHAPES:
        mbtiles_path = OUTPUT_DIR / f"{name}_{shape_name}_{GRID_SIZE}m.mbtiles"
        if not mbtiles_path.exists():
            continue

        with rasterio.open(mbtiles_path) as ds:
            arr = ds.read()
            src_crs = ds.crs
            src_bounds = ds.bounds
            print(f"{name} ({shape_name}): {ds.width}x{ds.height}, overviews={ds.overviews(1)}")

        img_hw = arr.transpose(1, 2, 0)
        w_lon, s_lat, e_lon, n_lat = transform_bounds(src_crs, "EPSG:4326", *src_bounds)
        center = [(s_lat + n_lat) / 2, (w_lon + e_lon) / 2]

        m = folium.Map(location=center, zoom_start=17, tiles="OpenStreetMap")
        folium.TileLayer(
            tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
            attr="Google Hybrid", name="Google Hybrid (online)",
        ).add_to(m)
        ImageOverlay(
            image=img_hw,
            bounds=[[s_lat, w_lon], [n_lat, e_lon]],
            opacity=1.0,
            name=f"{name} ({shape_name}) MBTiles",
        ).add_to(m)
        folium.LayerControl().add_to(m)
        display(m)